# 02 · Regressão de Retorno Esperado

Treina e avalia modelos de regressão para o problema "qual é o retorno esperado desta carteira dado seu vetor de pesos?"

**Modelos:**
- Regressão Linear (baseline)
- XGBoost Regressor (fine-tuning via GridSearchCV)

---

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[0]))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
from src.data.loader import load_carteiras_ml
from src.data.splitter import tabular_split
from src.features.portfolio_features import build_regression_dataset
from src.models.regressors import LinearRegressor
from src.models.xgboost_model import XGBoostModel
from src.evaluation.regression_metrics import evaluate_regressor, print_regression_report
from src.evaluation.model_comparator import compare_regressors
from src.visualization.regression_plots import plot_predicted_vs_actual, plot_residuals
from src.analysis.feature_importance import get_feature_importance, plot_feature_importance

## 1. Dados

In [ ]:
df = load_carteiras_ml()
X, y = build_regression_dataset(df, target='Retornos')

print('Estatísticas do target (retorno):')
print(y.describe().round(6))

In [ ]:
X_train, X_test, y_train, y_test = tabular_split(X, y, test_size=0.20, stratify=False)

## 2. Regressão Linear — Baseline

In [ ]:
linreg = LinearRegressor()
linreg.fit(X_train, y_train)

y_pred_lin = linreg.predict(X_test)
metrics_lin = evaluate_regressor(y_test, y_pred_lin, in_dollars=False)
print_regression_report(metrics_lin, model_name='Regressão Linear')

In [ ]:
# Coeficientes — interpretáveis como contribuição linear de cada peso
coef_df = pd.DataFrame({
    'feature':     list(X.columns),
    'coeficiente': linreg.coefficients,
}).sort_values('coeficiente', ascending=False)
print(f'Intercepto: {linreg.intercept:.6f}')
print('\nCoeficientes:')
coef_df

## 3. XGBoost Regressor — Fine-Tuning

In [ ]:
param_grid_quick = {
    'n_estimators':  [100, 300],
    'max_depth':     [3, 5],
    'learning_rate': [0.05, 0.1],
}

xgb_reg = XGBoostModel(
    task='regression',
    param_grid=param_grid_quick,
    cv_folds=5,
    scoring='r2',
)
xgb_reg.fit(X_train, y_train)

In [ ]:
print('Melhores parâmetros:')
for k, v in xgb_reg.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nMelhor R² em CV: {xgb_reg.best_score_:.4f}')

In [ ]:
y_pred_xgb = xgb_reg.predict(X_test)
metrics_xgb = evaluate_regressor(y_test, y_pred_xgb, in_dollars=False)
print_regression_report(metrics_xgb, model_name='XGBoost Regressor (Fine-Tuned)')

## 4. Comparação

In [ ]:
results = {
    'linear_regression': metrics_lin,
    'xgboost_reg':       metrics_xgb,
}
df_compare = compare_regressors(results)
df_compare

## 5. Análise Visual

In [ ]:
plot_predicted_vs_actual(
    y_test.values, y_pred_lin,
    title='Regressão Linear — Predito vs Real',
    save_as='reg_lin_pred_vs_actual.png',
)

In [ ]:
plot_predicted_vs_actual(
    y_test.values, y_pred_xgb,
    title='XGBoost Regressor — Predito vs Real',
    save_as='reg_xgb_pred_vs_actual.png',
)

In [ ]:
plot_residuals(y_test.values, y_pred_lin, title='Regressão Linear — Resíduos',
               save_as='reg_lin_residuos.png')

In [ ]:
plot_residuals(y_test.values, y_pred_xgb, title='XGBoost — Resíduos',
               save_as='reg_xgb_residuos.png')

## 6. Importância de Features (XGBoost)

In [ ]:
df_imp = get_feature_importance(xgb_reg.model, list(X.columns))
print(df_imp)
plot_feature_importance(df_imp, title='XGBoost Regressor — Feature Importance',
                         save_as='reg_xgb_feat_imp.png')

## 7. Conclusões

A regressão sobre os pesos da carteira é **um problema essencialmente linear** quando o target é o retorno esperado calculado pela fórmula `r = w · μ`. Por isso, é esperado que:

- A Regressão Linear apresente R² próximo de 1
- Os coeficientes sejam aproximadamente iguais aos retornos médios de cada ativo
- O XGBoost não traga ganho significativo (overkill para um problema linear)

Esse achado é em si um insight: o retorno é **estruturalmente linear** nos pesos, diferente do Sharpe (que envolve a covariância e é não-linear).